# Transformacja danych

In [31]:
import pandas as pd
import numpy as np

In [32]:
df = pd.read_csv('../dataset/dirty_cafe_sales_cleaned.csv')
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08 00:00:00
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16 00:00:00
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19 00:00:00
3,TXN_7034554,Salad,2.0,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27 00:00:00
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11 00:00:00


In [33]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
df = df[df['Transaction Date'].notna()].copy()
df[['Transaction Date']].head()

,Transaction Date
0,2023-09-08
1,2023-05-16
2,2023-07-19
3,2023-04-27
4,2023-06-11


In [34]:
df['Year'] = df['Transaction Date'].dt.year
df['Month'] = df['Transaction Date'].dt.month
df['Day'] = df['Transaction Date'].dt.day
df['Weekday'] = df['Transaction Date'].dt.weekday
df['week_start'] = df['Transaction Date'] - pd.to_timedelta(df['Transaction Date'].dt.weekday, unit='d')
df['YearWeek'] = df['week_start'].dt.strftime('%Y-%W')
df[['Year', 'Month', 'Day', 'Weekday', 'YearWeek']].head()


C:\Users\bkiel\AppData\Local\Temp\ipykernel_17160\2506095580.py:5: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  df['week_start'] = df['Transaction Date'] - pd.to_timedelta(df['Transaction Date'].dt.weekday, unit='d')


,Year,Month,Day,Weekday,YearWeek
0,2023,9,8,4,2023-36
1,2023,5,16,1,2023-20
2,2023,7,19,2,2023-29
3,2023,4,27,3,2023-17
4,2023,6,11,6,2023-23


In [35]:
df['Item'] = df['Item'].astype(str).str.strip().str.title()
df['Item'].head()


0    Coffee
1      Cake
2    Cookie
3     Salad
4    Coffee
Name: Item, dtype: str

In [36]:
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')
df[['Quantity', 'Price Per Unit', 'Total Spent']].head()


,Quantity,Price Per Unit,Total Spent
0,2.0,2.0,4.0
1,4.0,3.0,12.0
2,4.0,1.0,4.0
3,2.0,5.0,10.0
4,2.0,2.0,4.0


In [37]:
weekly_sales = (
    df
    .groupby(['YearWeek', 'Item', 'week_start'], as_index=False)
    .agg(
        weekly_quantity=('Quantity', 'sum'),
        weekly_revenue=('Total Spent', 'sum'),
        transaction_count=('Transaction ID', 'count')
    )
)
weekly_sales['avg_price_per_unit'] = np.where(
    weekly_sales['weekly_quantity'] > 0,
    weekly_sales['weekly_revenue'] / weekly_sales['weekly_quantity'],
    0
)

# Feature engineering for modeling
weekly_sales = weekly_sales.sort_values(['Item', 'week_start']).copy()
weekly_sales['month'] = weekly_sales['week_start'].dt.month
weekly_sales['week_number'] = weekly_sales['week_start'].dt.isocalendar().week
weekly_sales['lag_qty_1'] = weekly_sales.groupby('Item')['weekly_quantity'].shift(1)
weekly_sales['lag_qty_2'] = weekly_sales.groupby('Item')['weekly_quantity'].shift(2)
weekly_sales['lag_qty_3'] = weekly_sales.groupby('Item')['weekly_quantity'].shift(3)
weekly_sales['avg_qty_last_4_weeks'] = (
    weekly_sales.groupby('Item')['weekly_quantity']
    .shift(1)
    .rolling(4, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)
weekly_sales['next_qty'] = weekly_sales.groupby('Item')['weekly_quantity'].shift(-1)
weekly_sales['pct_change_next'] = (
    weekly_sales['next_qty'] - weekly_sales['weekly_quantity']
) / weekly_sales['weekly_quantity']
threshold = 0.10
weekly_sales['direction'] = np.where(
    weekly_sales['pct_change_next'] > threshold, 'increase',
    np.where(weekly_sales['pct_change_next'] < -threshold, 'decrease', 'similar')
)

weekly_sales.head()

,YearWeek,Item,week_start,weekly_quantity,weekly_revenue,transaction_count,avg_price_per_unit,month,week_number,lag_qty_1,lag_qty_2,lag_qty_3,avg_qty_last_4_weeks,next_qty,pct_change_next,direction
0,2022-52,Cake,2022-12-26,14.0,42.0,4,3.0,12,52,NaN,NaN,NaN,NaN,78.0,4.571429,increase
8,2023-01,Cake,2023-01-02,78.0,234.0,25,3.0,1,1,14.0,NaN,NaN,82.00,58.0,-0.256410,decrease
16,2023-02,Cake,2023-01-09,58.0,174.0,23,3.0,1,2,78.0,14.0,NaN,80.25,74.0,0.275862,increase
24,2023-03,Cake,2023-01-16,74.0,222.0,26,3.0,1,3,58.0,78.0,14.0,84.00,78.0,0.054054,similar
32,2023-04,Cake,2023-01-23,78.0,234.0,23,3.0,1,4,74.0,58.0,78.0,67.25,92.0,0.179487,increase


In [38]:
weekly_sales = weekly_sales.sort_values(['Item', 'week_start']).copy()
weekly_sales['month'] = weekly_sales['week_start'].dt.month
weekly_sales['week_number'] = weekly_sales['week_start'].dt.isocalendar().week
weekly_sales['lag_qty_1'] = weekly_sales.groupby('Item')['weekly_quantity'].shift(1)
weekly_sales['lag_qty_2'] = weekly_sales.groupby('Item')['weekly_quantity'].shift(2)
weekly_sales['lag_qty_3'] = weekly_sales.groupby('Item')['weekly_quantity'].shift(3)
weekly_sales['avg_qty_last_4_weeks'] = (
    weekly_sales.groupby('Item')['weekly_quantity']
    .shift(1)
    .rolling(4, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)
weekly_sales['next_qty'] = weekly_sales.groupby('Item')['weekly_quantity'].shift(-1)
weekly_sales['pct_change_next'] = (
    weekly_sales['next_qty'] - weekly_sales['weekly_quantity']
) / weekly_sales['weekly_quantity']
threshold = 0.10
weekly_sales['direction'] = np.where(
    weekly_sales['pct_change_next'] > threshold, 'increase',
    np.where(weekly_sales['pct_change_next'] < -threshold, 'decrease', 'similar')
)
weekly_sales_features = weekly_sales.dropna(
    subset=['lag_qty_1', 'lag_qty_2', 'lag_qty_3', 'avg_qty_last_4_weeks', 'next_qty']
).copy()
weekly_sales_features.head()


,YearWeek,Item,week_start,weekly_quantity,weekly_revenue,transaction_count,avg_price_per_unit,month,week_number,lag_qty_1,lag_qty_2,lag_qty_3,avg_qty_last_4_weeks,next_qty,pct_change_next,direction
24,2023-03,Cake,2023-01-16,74.0,222.0,26,3.0,1,3,58.0,78.0,14.0,84.00,78.0,0.054054,similar
32,2023-04,Cake,2023-01-23,78.0,234.0,23,3.0,1,4,74.0,58.0,78.0,67.25,92.0,0.179487,increase
40,2023-05,Cake,2023-01-30,92.0,276.0,29,3.0,1,5,78.0,74.0,58.0,87.50,82.0,-0.108696,decrease
48,2023-06,Cake,2023-02-06,82.0,246.0,29,3.0,2,6,92.0,78.0,74.0,90.50,76.0,-0.073171,similar
56,2023-07,Cake,2023-02-13,76.0,228.0,22,3.0,2,7,82.0,92.0,78.0,52.00,53.0,-0.302632,decrease


In [39]:
df.to_csv('../dataset/transformacja_intermediate.csv', index=False)
weekly_sales.to_csv('../dataset/weekly_sales_by_item.csv', index=False)
weekly_sales_features = weekly_sales.dropna(
    subset=['lag_qty_1', 'lag_qty_2', 'lag_qty_3', 'avg_qty_last_4_weeks', 'next_qty']
).copy()
weekly_sales_features.to_csv('../dataset/weekly_sales_by_item_features.csv', index=False)
print(weekly_sales.head())
print(df.head())

   YearWeek  Item week_start  weekly_quantity  weekly_revenue  \
0   2022-52  Cake 2022-12-26             14.0            42.0   
8   2023-01  Cake 2023-01-02             78.0           234.0   
16  2023-02  Cake 2023-01-09             58.0           174.0   
24  2023-03  Cake 2023-01-16             74.0           222.0   
32  2023-04  Cake 2023-01-23             78.0           234.0   

    transaction_count  avg_price_per_unit  month  week_number  lag_qty_1  \
0                   4                 3.0     12           52        NaN   
8                  25                 3.0      1            1       14.0   
16                 23                 3.0      1            2       78.0   
24                 26                 3.0      1            3       58.0   
32                 23                 3.0      1            4       74.0   

    lag_qty_2  lag_qty_3  avg_qty_last_4_weeks  next_qty  pct_change_next  \
0         NaN        NaN                   NaN      78.0         4.571429  